# KVO_04 — Final multisite UAV U-Net and probability maps

Production model using all four sites and crowns ≥5 m². Stage A uses spatial 100 m validation blocks only to select epoch count. Stage B refits a fresh U-Net for that fixed epoch count on all eligible crown labels. It then generates continuous wall-to-wall Musanga probability rasters for all four orthomosaics. KVO_03 remains the independent LOSO validation.


In [ ]:
!pip -q install geopandas rasterio pyogrio segmentation-models-pytorch scikit-learn matplotlib tqdm
from pathlib import Path
import json, hashlib, random, shutil, time, warnings, gc
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
from sklearn.metrics import roc_auc_score, average_precision_score
from tqdm.auto import tqdm
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from google.colab import drive
warnings.filterwarnings('ignore')
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'; torch.backends.cudnn.benchmark=True
print('DEVICE:',DEVICE); print(torch.cuda.get_device_name(0) if DEVICE=='cuda' else '')

In [ ]:
DRIVE_ROOT=Path('/content/drive'); MYDRIVE=DRIVE_ROOT/'MyDrive'
def healthy():
    try: return MYDRIVE.exists() and any(MYDRIVE.iterdir())
    except OSError: return False
if not healthy(): drive.mount(str(DRIVE_ROOT),force_remount=True)
ROOT=MYDRIVE/'Colab Notebooks'/'Musanga'; GEO=ROOT/'data'/'Geo_Ref'; AD=GEO/'Annotations'
OUT=ROOT/'KVO_work'/'KVO_04_final_multisite'; CACHE=OUT/'cache'; MAPS=OUT/'probability_maps'
for p in [OUT,CACHE,MAPS]: p.mkdir(parents=True,exist_ok=True)
SITES={
 'MOS_1_2':{'raster':GEO/'MOS_1_2_kvo.tif','annotations':AD/'Tree_Annotations_MOS_1_2_kvo.shp'},
 'Yoko':{'raster':GEO/'Yoko_kvo_georef.tif','annotations':AD/'Tree_Annotations_Yoko_kvo_georef.shp'},
 'Baego':{'raster':GEO/'Baego_georef.tif','annotations':AD/'Tree_Annotation_Baego.shp'},
 'Yangambi':{'raster':GEO/'Yangambi_georef.tif','annotations':AD/'Tree_Annotation_Yangambi_georef.shp'}}
for s,d in SITES.items(): assert d['raster'].exists() and d['annotations'].exists(),s
PATCH=256; STRIDE=256; IGNORE=255; BLOCK_M=100.; VAL_FRAC=.20; MIN_CROWN_AREA_M2=5.0
MEAN=np.array([0.485,0.456,0.406],np.float32)[:,None,None]; STD=np.array([0.229,0.224,0.225],np.float32)[:,None,None]
FORCE_CACHE=False; FORCE_TUNE=False; FORCE_REFIT=False; FORCE_MAPS=False
print('OUT:',OUT)


## 1 — Audit filtered crowns


In [ ]:
ANNS={}; audit=[]
for site,d in SITES.items():
    with rasterio.open(d['raster']) as src: crs=src.crs; bounds=src.bounds; gsd=(abs(src.transform.a),abs(src.transform.e)); dtype=src.dtypes[0]
    assert dtype=='uint8',f'{site}: expected uint8 RGB, got {dtype}'
    g=gpd.read_file(d['annotations']); assert 'Class_ID' in g.columns
    if g.crs!=crs: g=g.to_crs(crs)
    g=g[g.geometry.notna() & ~g.geometry.is_empty].copy(); g['class_id_num']=pd.to_numeric(g.Class_ID,errors='coerce'); g=g[g.class_id_num.notna()].copy(); g['class_id_num']=g.class_id_num.astype(int)
    g['target']=(g.class_id_num==1).astype(np.uint8)
    if g.crs is None or not g.crs.is_projected: raise ValueError(f'{site}: projected CRS required for m² crown filtering; got {g.crs}')
    g['crown_area_m2']=g.geometry.area.astype(float)
    n_before=len(g); mus_before=int(g.target.sum()); other_before=int((1-g.target).sum())
    g=g[g.crown_area_m2>=MIN_CROWN_AREA_M2].copy().reset_index(drop=True)
    g['crown_id']=np.arange(len(g),dtype=np.int64); c=g.geometry.centroid
    g['block_id']=np.floor((c.x-bounds.left)/BLOCK_M).astype(int).astype(str)+'_'+np.floor((c.y-bounds.bottom)/BLOCK_M).astype(int).astype(str)
    ANNS[site]=g; audit.append({'site':site,'min_crown_area_m2':MIN_CROWN_AREA_M2,'n_before':n_before,'removed_lt_5m2':n_before-len(g),'n_crowns':len(g),'musanga':int(g.target.sum()),'other':int((1-g.target).sum()),'musanga_removed':mus_before-int(g.target.sum()),'other_removed':other_before-int((1-g.target).sum()),'blocks':g.block_id.nunique(),'gsd_x':gsd[0],'gsd_y':gsd[1]})
audit_df=pd.DataFrame(audit); display(audit_df); audit_df.to_csv(OUT/'site_audit.csv',index=False)

def train_val_assignment(g,seed):
    blocks=g.block_id.unique().copy(); rng=np.random.default_rng(seed)
    for attempt in range(1000):
        b=blocks.copy(); rng.shuffle(b); nv=max(1,int(round(VAL_FRAC*len(b)))); vb=set(b[:nv]); sp=np.where(g.block_id.isin(vb),'val','train')
        if all(g.loc[sp==q,'target'].nunique()==2 for q in ['train','val']): return pd.Series(sp,index=g.index)
    raise RuntimeError('Could not make two-class train/val split')

## 2 — Reusable all-site caches


In [ ]:
def train_val_assignment(g,seed):
    blocks=g.block_id.unique().copy(); rng=np.random.default_rng(seed)
    for _ in range(1000):
        b=blocks.copy(); rng.shuffle(b); nv=max(1,int(round(VAL_FRAC*len(b)))); vb=set(b[:nv])
        sp=np.where(g.block_id.isin(vb),'val','train')
        if all(g.loc[sp==q,'target'].nunique()==2 for q in ['train','val']): return pd.Series(sp,index=g.index)
    raise RuntimeError('Could not make two-class split')

def build_cache(mode='tune',force=False):
    assert mode in ['tune','all']; split_ann={}
    for i,s in enumerate(SITES):
        g=ANNS[s].copy(); g['split']=train_val_assignment(g,SEED+100*i) if mode=='tune' else 'train'; split_ann[s]=g
    payload={'mode':mode,'patch':PATCH,'block_m':BLOCK_M,'val_frac':VAL_FRAC,'min_area':MIN_CROWN_AREA_M2,
      'files':{s:(SITES[s]['raster'].stat().st_size,SITES[s]['annotations'].stat().st_size,int(SITES[s]['annotations'].stat().st_mtime)) for s in SITES}}
    fp=hashlib.sha1(json.dumps(payload,sort_keys=True).encode()).hexdigest()[:12]; path=CACHE/f'{mode}_patches_{fp}.npz'
    if path.exists() and not force:
        print('Reusing cache:',path); d=np.load(path); return d['X'],d['Y'],d['S'],path
    X=[]; Y=[]; SS=[]; tmp=Path('/content/musanga_kvo04'); tmp.mkdir(exist_ok=True)
    for site in SITES:
        g=split_ann[site]; rp=SITES[site]['raster']
        with rasterio.open(rp) as src:
            H,W=src.height,src.width; splits=['train','val'] if mode=='tune' else ['train']; mps={}
            for sp in splits:
                mp=tmp/f'{mode}_{site}_{sp}.tif'; prof=src.profile.copy(); prof.update(count=1,dtype='uint8',nodata=IGNORE,compress='lzw',tiled=True)
                a=g[g.split==sp]
                m=rasterize([(z,int(y)) for z,y in zip(a.geometry,a.target)],out_shape=(H,W),transform=src.transform,fill=IGNORE,dtype='uint8')
                with rasterio.open(mp,'w',**prof) as dst: dst.write(m,1)
                del m; gc.collect(); mps[sp]=mp
            rr={sp:rasterio.open(mp) for sp,mp in mps.items()}
            try:
                for r0 in tqdm(range(0,H,PATCH),desc=f'{mode}: cache {site}'):
                    for c0 in range(0,W,PATCH):
                        hh=min(PATCH,H-r0); ww=min(PATCH,W-c0); w=Window(c0,r0,ww,hh)
                        mm={sp:rr[sp].read(1,window=w) for sp in splits}; hit={sp:np.any(mm[sp]!=IGNORE) for sp in splits}
                        if mode=='tune':
                            if hit['train']==hit['val']: continue
                            sp='train' if hit['train'] else 'val'; m=mm[sp]; sc=0 if sp=='train' else 1
                        else:
                            if not hit['train']: continue
                            m=mm['train']; sc=0
                        arr=src.read([1,2,3],window=w,boundless=True,fill_value=0,out_shape=(3,PATCH,PATCH))
                        y=np.full((PATCH,PATCH),IGNORE,np.uint8); y[:hh,:ww]=m
                        X.append(arr.astype(np.uint8)); Y.append(y); SS.append(sc)
            finally:
                for q in rr.values(): q.close()
                for mp in mps.values(): mp.unlink(missing_ok=True)
    X=np.stack(X); Y=np.stack(Y); SS=np.asarray(SS,np.uint8); np.savez_compressed(path,X=X,Y=Y,S=SS)
    print('Saved:',path,'train/val:',int((SS==0).sum()),int((SS==1).sum()))
    return X,Y,SS,path


## 3 — Select epoch count, then refit on all four sites


In [ ]:
class DS(Dataset):
    def __init__(self,X,Y,S,code=0,aug=False): self.X=X; self.Y=Y; self.idx=np.where(S==code)[0]; self.aug=aug
    def __len__(self): return len(self.idx)
    def __getitem__(self,j):
        i=self.idx[j]; x=self.X[i].astype(np.float32)/255.; y=self.Y[i].copy()
        if self.aug:
            if np.random.rand()<.5: x=x[:,:,::-1].copy(); y=y[:,::-1].copy()
            if np.random.rand()<.5: x=x[:,::-1,:].copy(); y=y[::-1,:].copy()
            k=np.random.randint(4); x=np.rot90(x,k,axes=(1,2)).copy(); y=np.rot90(y,k).copy(); x=np.clip(x*np.random.uniform(.9,1.1),0,1)
        return torch.from_numpy((x-MEAN)/STD),torch.from_numpy(y.astype(np.int64))
def model_new(): return smp.Unet(encoder_name='resnet18',encoder_weights='imagenet',in_channels=3,classes=1).to(DEVICE)
def score(model,dl):
    yy=[]; pp=[]; model.eval()
    with torch.inference_mode():
        for x,y in dl:
            x=x.to(DEVICE); y=y.to(DEVICE)
            with torch.amp.autocast('cuda',enabled=DEVICE=='cuda'): p=torch.sigmoid(model(x).squeeze(1))
            v=y!=IGNORE; yy.append(y[v].cpu().numpy()); pp.append(p[v].float().cpu().numpy())
    y=np.concatenate(yy); p=np.concatenate(pp); return roc_auc_score(y,p),average_precision_score(y,p)
def fit(model,dl,n,va=None,ck=None,hp=None,early=False):
    opt=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4); scaler=torch.amp.GradScaler('cuda',enabled=DEVICE=='cuda')
    best=-1; be=None; wait=0; hist=[]
    for ep in range(1,n+1):
        model.train(); ls=[]
        for x,y in dl:
            x=x.to(DEVICE); y=y.to(DEVICE); opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda',enabled=DEVICE=='cuda'):
                z=model(x).squeeze(1); v=y!=IGNORE; loss=nn.functional.binary_cross_entropy_with_logits(z[v],y[v].float())
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); ls.append(loss.item())
        row={'epoch':ep,'loss':float(np.mean(ls))}
        if va is not None:
            auc,ap=score(model,va); row.update(val_auc=auc,val_ap=ap); print(f'tune ep {ep:02d} loss={row["loss"]:.4f} AUC={auc:.4f} AP={ap:.4f}')
            if ap>best: best=ap; be=ep; wait=0; torch.save(model.state_dict(),ck)
            else: wait+=1
        else: print(f'refit ep {ep:02d}/{n} loss={row["loss"]:.4f}')
        hist.append(row)
        if hp: pd.DataFrame(hist).to_csv(hp,index=False)
        if early and wait>=6: break
    return be

Xt,Yt,St,tc=build_cache('tune',FORCE_CACHE); tid=tc.stem.replace('tune_patches_','')
bj=OUT/f'best_epoch_{tid}.json'; tck=OUT/f'tune_best_{tid}.pt'; th=OUT/f'tune_history_{tid}.csv'; bs=16 if DEVICE=='cuda' else 4
tr=DataLoader(DS(Xt,Yt,St,0,True),batch_size=bs,shuffle=True,num_workers=2,pin_memory=DEVICE=='cuda',persistent_workers=True,prefetch_factor=2)
va=DataLoader(DS(Xt,Yt,St,1,False),batch_size=bs,shuffle=False,num_workers=2,pin_memory=DEVICE=='cuda',persistent_workers=True,prefetch_factor=2)
if bj.exists() and tck.exists() and not FORCE_TUNE:
    BEST_EPOCH=int(json.loads(bj.read_text())['best_epoch']); print('Reusing selected epoch:',BEST_EPOCH)
else:
    m=model_new(); BEST_EPOCH=int(fit(m,tr,30,va,tck,th,True)); bj.write_text(json.dumps({'best_epoch':BEST_EPOCH,'selection':'max spatial-validation AP'},indent=2)); del m
del Xt,Yt,St,tr,va; gc.collect(); torch.cuda.empty_cache()
print('Selected refit epochs:',BEST_EPOCH)

Xa,Ya,Sa,ac=build_cache('all',FORCE_CACHE); aid=ac.stem.replace('all_patches_','')+f'_ep{BEST_EPOCH:02d}'
FINAL_CK=OUT/f'final_unet_resnet18_{aid}.pt'; FH=OUT/f'final_refit_history_{aid}.csv'
if FINAL_CK.exists() and FH.exists() and not FORCE_REFIT:
    print('Existing final model — SKIPPING REFIT'); final_model=model_new(); final_model.load_state_dict(torch.load(FINAL_CK,map_location=DEVICE))
else:
    dl=DataLoader(DS(Xa,Ya,Sa,0,True),batch_size=bs,shuffle=True,num_workers=2,pin_memory=DEVICE=='cuda',persistent_workers=True,prefetch_factor=2)
    final_model=model_new(); fit(final_model,dl,BEST_EPOCH,None,None,FH,False); torch.save(final_model.state_dict(),FINAL_CK); print('Saved:',FINAL_CK)
del Xa,Ya,Sa; gc.collect(); torch.cuda.empty_cache()


## 3B — Canonical ROI construction from crown annotations (10 m aggregation)

**CPU only.** Replaces the project-wide `ROI.shp` assumption with one **site-specific, reproducible ROI construction rule** for MOS_1_2, Yoko, Baego, Yangambi and Luki. Read each site's original annotation polygons (both Musanga and other tree crowns), group polygons separated by ≤10 m using connected components, and export each group's **union of 5 m buffers**, exactly the method used in Luki Block 71. No previously drawn ROI is read. Inspect group counts, geometry and site-specific footprint; derived groups are **not surveyed plot boundaries**.

**Run this cell first after reopening the notebook**, before Sections 4/5 or Section 7. It is self-contained and does not run the model. Saved outputs in `KVO_work/KVO_04_final_multisite/annotation_derived_rois_10m/`. **Existing probability rasters are not changed** by this section.

**Data governance:** all crown polygons define ROI coverage; filtering to Class_ID=1 or 5 m² would incorrectly omit non-Musanga/small-crown regions. Existing probability maps previously masked with a different ROI cannot be expanded by simple remasking; compare valid-pixel coverage before replacing products.

In [ ]:
# KVO_04 SECTION 3B — Canonical annotation-derived 10m ROIs for all sites.
# CPU ONLY. Execute after Section 2 (ROOT/GEO/AD/SITES), or independently:
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio,json,hashlib
from shapely.geometry import Polygon,MultiPolygon,box
from shapely.ops import unary_union
from shapely.validation import make_valid
from shapely.strtree import STRtree
from IPython.display import display
if "ROOT" not in globals():
    from google.colab import drive
    rootb=Path("/content/drive/MyDrive")
    if not rootb.exists():drive.mount("/content/drive")
    ROOT=rootb/"Colab Notebooks"/"Musanga"
GEO=ROOT/"data"/"Geo_Ref"; AD=GEO/"Annotations"
OUT=ROOT/"KVO_work"/"KVO_04_final_multisite"
MAPS=OUT/"probability_maps"
ROI_DIR=OUT/"annotation_derived_rois_10m"
ROI_DIR.mkdir(parents=True,exist_ok=True)
if "SITES" not in globals():
    SITES={
     "MOS_1_2":{"raster":GEO/"MOS_1_2_kvo.tif","annotations":AD/"Tree_Annotations_MOS_1_2_kvo.shp"},
     "Yoko":{"raster":GEO/"Yoko_kvo_georef.tif","annotations":AD/"Tree_Annotations_Yoko_kvo_georef.shp"},
     "Baego":{"raster":GEO/"Baego_georef.tif","annotations":AD/"Tree_Annotation_Baego.shp"},
     "Yangambi":{"raster":GEO/"Yangambi_georef.tif","annotations":AD/"Tree_Annotation_Yangambi_georef.shp"}}
SITES_ROI={k:dict(v) for k,v in SITES.items()}
SITES_ROI["Luki"]={"raster":GEO/"Luki.tif",
     "annotations":AD/"Tree_Annotation_Luki.gpkg"}
AGG_DISTANCE_M=10.; ROI_OUTER_BUFFER_M=5.
_ROI_CACHE={}
def _valid_poly_roi(g):
    if g is None or g.is_empty:return Polygon()
    g=make_valid(g)
    if isinstance(g,(Polygon,MultiPolygon)):return g
    polys=[x for x in getattr(g,"geoms",[]) if isinstance(x,(Polygon,MultiPolygon))]
    return unary_union(polys) if polys else Polygon()
def aggregate_annotation_rois(site,export=True):
    """10m-connected crown footprints, exactly same 5m-buffer principle as Luki Block71.
    No external ROI file is read. ALL valid tree polygons (both Class_ID classes)
    are used. ROIs are diagnostic buffered footprints, not surveyed plot boundaries.
    """
    site=str(site)
    if site in _ROI_CACHE:return _ROI_CACHE[site].copy()
    assert site in SITES_ROI,f"Unknown site {site}"
    spec=SITES_ROI[site]
    assert Path(spec["annotations"]).exists(),spec["annotations"]
    assert Path(spec["raster"]).exists(),spec["raster"]
    with rasterio.open(spec["raster"]) as ds:
        assert ds.crs is not None and ds.crs.is_projected
        assert ds.crs.linear_units.lower() in ("metre","meter","metres","meters"),(
            site,ds.crs.linear_units)
        dest_crs=ds.crs
        footprint=box(*ds.bounds)
    if Path(spec["annotations"]).suffix.lower()==".gpkg":
        layerlist=gpd.list_layers(spec["annotations"])
        spatial=layerlist[layerlist.geometry_type.notna()]
        assert len(spatial)==1,(site,spatial)
        ann=gpd.read_file(spec["annotations"],layer=str(spatial.iloc[0]["name"]))
    else:
        with rasterio.Env(SHAPE_RESTORE_SHX="YES"):
            ann=gpd.read_file(spec["annotations"])
    assert ann.crs is not None and len(ann)>0,(site,"missing CRS or empty annotations")
    original_count=len(ann)
    ann=ann[ann.geometry.notna()&~ann.geometry.is_empty].copy()
    ann=ann.to_crs(dest_crs)
    ann["geometry"]=ann.geometry.map(_valid_poly_roi)
    ann=ann[~ann.geometry.is_empty&(ann.geometry.area>0)].reset_index(drop=True)
    assert len(ann)>0,(site,"no positive-area crown polygons")
    geoms=list(ann.geometry)
    tree=STRtree(geoms)
    parents=list(range(len(geoms)))
    def find(i):
        while parents[i]!=i:
            parents[i]=parents[parents[i]]
            i=parents[i]
        return i
    def join(i,j):
        ri,rj=find(i),find(j)
        if ri!=rj:parents[max(ri,rj)]=min(ri,rj)
    for i,g in enumerate(geoms):
        for j in tree.query(g.buffer(AGG_DISTANCE_M)):
            j=int(j)
            if j>i and g.distance(geoms[j])<=AGG_DISTANCE_M+1e-7:
                join(i,j)
    components={}
    for i in range(len(geoms)):
        components.setdefault(find(i),[]).append(i)
    groups=sorted(components.values(),key=lambda ids:(
        -len(ids),-sum(geoms[j].area for j in ids),min(ids)))
    rows=[]
    for k,ids in enumerate(groups,1):
        poly=_valid_poly_roi(unary_union(
            [geoms[j].buffer(ROI_OUTER_BUFFER_M,join_style=1) for j in ids]))
        assert poly.is_valid and all(poly.covers(geoms[j]) for j in ids)
        rows.append(dict(plot_id=f"{site.upper()}_{k:03d}",
            n_crowns=len(ids),crown_m2=float(sum(geoms[j].area for j in ids)),
            roi_m2=float(poly.area),raster_overlap_pct=
                float(poly.intersection(footprint).area/max(poly.area,1e-12)*100),
            geometry=poly))
    out=gpd.GeoDataFrame(rows,geometry="geometry",crs=dest_crs)
    assert int(out.n_crowns.sum())==len(geoms)
    # Different components must remain independent, even if their 5m buffers touch.
    assert out.is_valid.all()
    assert out.raster_overlap_pct.min()>99.,(
        site,"ROI extends beyond orthomosaic; inspect edge groups first")
    if export:
        dest=ROI_DIR/f"{site}_ROI_10m.gpkg"
        # Never read from this file; only export for mapping/QC.
        if dest.exists():dest.unlink()
        out.to_file(dest,layer="roi_10m",driver="GPKG")
    print(f"{site}: {original_count} input features, {len(geoms)} valid crowns, "
          f"{len(out)} groups; largest {int(out.n_crowns.max())}, "
          f"smallest {int(out.n_crowns.min())}; raster overlap min "
          f"{out.raster_overlap_pct.min():.2f}%")
    _ROI_CACHE[site]=out
    return out.copy()

# Safe audit only: does NOT touch saved probability GeoTIFFs.
audit_roi=[]
for s in SITES_ROI:
    try:
        q=aggregate_annotation_rois(s)
        audit_roi.append(dict(site=s,crowns=int(q.n_crowns.sum()),groups=len(q),
              min_group=int(q.n_crowns.min()),max_group=int(q.n_crowns.max()),
              total_roi_ha=float(q.roi_m2.sum()/10000),
              min_raster_overlap_pct=float(q.raster_overlap_pct.min())))
    except Exception as error:
        print(f"ROI derivation FAILED for {s}: {type(error).__name__}: {error}")
        raise
audit_roi=pd.DataFrame(audit_roi)
audit_roi.to_csv(ROI_DIR/"site_roi_aggregation_summary.csv",index=False)
display(audit_roi.round(2))
print("SECTION 3B PASS — all ROIs computed from individual tree annotations; no external ROI polygons read.")


## 4 — Inference using annotation-derived site ROIs

The canonical ROI mask is computed using **Section 3B** directly from each site's crown annotation polygons. No `ROI.shp` is read. UInt16 encoding and 512/64 core/halo preprocessing are unchanged. Existing maps are **not automatically regenerated**; if newly derived ROIs extend beyond their existing valid pixel coverage, full inference of those missing pixels is needed before claiming the new ROI is completely mapped.

In [ ]:
CORE=512; HALO=64; SCALE=10000; OUT_NODATA=65535
assert "aggregate_annotation_rois" in globals(), "Run Section 3B first: canonical site-derived ROIs."

def roi_for_raster(site):
    # Recomputed from original crown annotation polygons, never a supplied ROI file.
    return aggregate_annotation_rois(site,export=True)

def output_ok(p,sp):
    if not p.exists(): return False
    try:
        with rasterio.open(sp) as a,rasterio.open(p) as b:
            return b.width==a.width and b.height==a.height and b.crs==a.crs and b.transform==a.transform and b.dtypes[0]=='uint16'
    except: return False

def infer_site(model,site,force=False):
    sp=SITES[site]['raster']; op=MAPS/f'{site}_musanga_probability_uint16.tif'
    if output_ok(op,sp) and not force:
        print('Existing map — SKIPPING inference; its mask may differ from newly derived ROIs (audit Section 5):',op)
        return op
    roi=roi_for_raster(site)
    if roi.empty: raise ValueError(f'{site}: no annotation-derived ROI overlaps raster')
    shapes=[(geom,1) for geom in roi.geometry]
    lp=Path('/content')/op.name; lp.unlink(missing_ok=True); model.eval()
    with rasterio.open(sp) as src:
        prof=src.profile.copy(); prof.update(count=1,dtype='uint16',nodata=OUT_NODATA,compress='deflate',predictor=2,tiled=True,bigtiff='YES')
        with rasterio.open(lp,'w',**prof) as dst:
            # Initialize full raster as NoData so windows outside ROI need no inference.
            for r0 in range(0,src.height,CORE):
                for c0 in range(0,src.width,CORE):
                    hh=min(CORE,src.height-r0); ww=min(CORE,src.width-c0)
                    dst.write(np.full((hh,ww),OUT_NODATA,np.uint16),1,window=Window(c0,r0,ww,hh))
            for r0 in tqdm(range(0,src.height,CORE),desc=f'MAP {site}'):
                for c0 in range(0,src.width,CORE):
                    hh=min(CORE,src.height-r0); ww=min(CORE,src.width-c0); cw=Window(c0,r0,ww,hh)
                    roi_mask=rasterize(shapes,out_shape=(hh,ww),transform=src.window_transform(cw),fill=0,dtype='uint8',all_touched=False)>0
                    if not roi_mask.any(): continue
                    win=Window(c0-HALO,r0-HALO,ww+2*HALO,hh+2*HALO)
                    a=src.read([1,2,3],window=win,boundless=True,fill_value=0).astype(np.float32)/255.
                    ph=(32-a.shape[1]%32)%32; pw=(32-a.shape[2]%32)%32
                    if ph or pw: a=np.pad(a,((0,0),(0,ph),(0,pw)))
                    x=torch.from_numpy((a-MEAN)/STD).unsqueeze(0).to(DEVICE)
                    with torch.inference_mode(),torch.amp.autocast('cuda',enabled=DEVICE=='cuda'):
                        p=torch.sigmoid(model(x).squeeze()).float().cpu().numpy()
                    p=p[:int(win.height),:int(win.width)][HALO:HALO+hh,HALO:HALO+ww]
                    # ROI is authoritative; dataset mask additionally excludes declared source NoData.
                    valid=roi_mask & (src.dataset_mask(window=cw)>0)
                    o=np.full((hh,ww),OUT_NODATA,np.uint16)
                    o[valid]=np.rint(np.clip(p[valid],0,1)*SCALE).astype(np.uint16)
                    dst.write(o,1,window=cw)
            dst.update_tags(MUSANGA_SCALE=SCALE,MODEL=FINAL_CK.name,MIN_CROWN_AREA_M2=MIN_CROWN_AREA_M2,
                            TRAINING_SITES=','.join(SITES),ROI_MASK=f'computed from {SITES[site]["annotations"]}; 10m connected / 5m buffer')
    shutil.copy2(lp,op); lp.unlink(missing_ok=True); print('Saved:',op); return op

maps={s:infer_site(final_model,s,FORCE_MAPS) for s in SITES}


## 5 — Audit and optionally remask existing probabilities using calculated ROIs (CPU-only)

Run **Section 3B** first. Default is **QA ONLY**: calculate how much of each newly defined annotation-based ROI has existing valid probabilities, and how much valid area lies outside the new ROIs. The workflow will **not overwrite** any existing map by default.

If and only if every new ROI pixel already has a valid probability, an explicit opt-in `APPLY_ROI_REMASK=True` permits masking outside the ROI with a backup. If some new ROI pixels are missing, simple remasking cannot fill them; re-run frozen model inference for those areas. Backup and abort on incomplete overlap.

In [ ]:
# SECTION 5 — existing raster coverage audit; optional SAFE remasking.
# Run Section 3B first. Default never modifies any probability file.
import shutil
from rasterio.features import rasterize
from rasterio.windows import Window
from tqdm.auto import tqdm
assert "aggregate_annotation_rois" in globals(), "Run Section 3B first."
MAPS=OUT/"probability_maps"
OUT_NODATA=65535
ROI_BLOCK=2048
APPLY_ROI_REMASK=False   # Manually opt in ONLY after QA says missing_inside_roi=0

def audit_probability_roi(site):
    path=MAPS/f"{site}_musanga_probability_uint16.tif"
    srcpath=SITES_ROI[site]["raster"]
    assert path.exists(),f"Probability file missing: {path}"
    roi=aggregate_annotation_rois(site,export=True)
    candidates=list(roi.geometry)
    pixels=inside_roi=valid_inside=missing_inside=outside_valid=0
    with rasterio.open(srcpath) as src,rasterio.open(path) as pr:
        assert (src.width,src.height,src.transform,src.crs)==(
            pr.width,pr.height,pr.transform,pr.crs),site
        assert pr.nodata==OUT_NODATA and pr.dtypes[0]=="uint16"
        # Only count valid source image pixels; outside RGB coverage is not a missing prediction.
        for _,w in pr.block_windows(1):
            rect=box(*rasterio.windows.bounds(w,pr.transform))
            related=[g for g in candidates if g.intersects(rect)]
            valid=pr.read(1,window=w)!=OUT_NODATA
            source_ok=src.dataset_mask(window=w)>0
            if related:
                inroi=rasterize([(g,1) for g in related],out_shape=(int(w.height),int(w.width)),
                      transform=src.window_transform(w),fill=0,dtype="uint8")>0
            else:inroi=np.zeros(valid.shape,dtype=bool)
            inside=inroi&source_ok
            pixels+=int(source_ok.sum())
            inside_roi+=int(inside.sum())
            valid_inside+=int((inside&valid).sum())
            missing_inside+=int((inside&~valid).sum())
            outside_valid+=int((~inroi&valid).sum())
    return dict(site=site,source_valid_px=pixels,roi_source_valid_px=inside_roi,
                existing_valid_inside_roi=valid_inside,missing_inside_roi=missing_inside,
                existing_valid_outside_roi=outside_valid,
                coverage_pct=100*valid_inside/max(inside_roi,1))

audit_rows=[]
for s in SITES_ROI:
    p=MAPS/f"{s}_musanga_probability_uint16.tif"
    if p.exists():
        result=audit_probability_roi(s);audit_rows.append(result)
    else:print(s,"probability map does not exist; skipping coverage audit.")
roi_coverage_qa=pd.DataFrame(audit_rows)
if len(roi_coverage_qa):
    display(roi_coverage_qa)
    roi_coverage_qa.to_csv(ROI_DIR/"existing_probability_vs_new_rois.csv",index=False)
if APPLY_ROI_REMASK:
    assert len(roi_coverage_qa)>0,"No existing probability maps audited."
    incomplete=roi_coverage_qa[roi_coverage_qa.missing_inside_roi>0]
    assert incomplete.empty,("Remasking cannot create predictions inside expanded ROIs; "
                              "re-run inference for these sites:",incomplete.site.tolist())
    for site in roi_coverage_qa.site:
        path=MAPS/f"{site}_musanga_probability_uint16.tif"
        rois=list(aggregate_annotation_rois(site).geometry)
        temp=Path("/content")/f"{site}_calculated_roi_masked.tmp.tif"
        if temp.exists():temp.unlink()
        with rasterio.open(path) as pr:
            profile=pr.profile.copy()
            with rasterio.open(temp,"w",**profile) as dst:
                for _,w in pr.block_windows(1):
                    data=pr.read(1,window=w)
                    rect=box(*rasterio.windows.bounds(w,pr.transform))
                    near=[g for g in rois if g.intersects(rect)]
                    if near:
                        mask=rasterize([(g,1) for g in near],
                            out_shape=data.shape,transform=pr.window_transform(w),
                            fill=0,dtype="uint8")>0
                        data[~mask]=OUT_NODATA
                    else:data[:]=OUT_NODATA
                    dst.write(data,1,window=w)
                tags=pr.tags()
                tags.update(ROI_METHOD="10m connected annotation crown grouping / 5m buffered union",
                            ROI_SOURCE=str(SITES_ROI[site]["annotations"]))
                dst.update_tags(**tags)
        backup=path.with_name(path.stem+".before_calculated_roi.tif")
        assert not backup.exists(),f"Refusing overwrite existing backup: {backup}"
        shutil.copy2(path,backup)
        shutil.copy2(temp,path)
        temp.unlink()
        print("SAFE remask complete:",site,"| backup:",backup)
else:
    print("SECTION 5 QA ONLY — existing probability GeoTIFFs remain untouched.")


## 6 — Manifest and sampled QA


In [ ]:
rows=[]
for s,p in maps.items():
    with rasterio.open(p) as q:
        a=q.read(1,out_shape=(max(1,q.height//20),max(1,q.width//20))); v=a!=OUT_NODATA
        rows.append({'site':s,'path':str(p),'width':q.width,'height':q.height,'valid_sample_pixels':int(v.sum()),
                     'mean_probability_sample':float((a[v]/SCALE).mean()) if v.any() else np.nan,
                     'p95_probability_sample':float(np.quantile(a[v]/SCALE,.95)) if v.any() else np.nan})
qa=pd.DataFrame(rows); display(qa); qa.to_csv(OUT/'probability_map_QA.csv',index=False)
manifest={'model':str(FINAL_CK),'best_epoch':BEST_EPOCH,'minimum_crown_area_m2':MIN_CROWN_AREA_M2,'training_sites':list(SITES),
          'encoding':'uint16 0..10000 = probability 0..1; 65535 = NoData','maps':{s:str(p) for s,p in maps.items()},
          'validation_note':'Independent site-generalization evidence is KVO_03 LOSO; KVO_04 is final all-data production fit.'}
(OUT/'production_manifest.json').write_text(json.dumps(manifest,indent=2)); print(json.dumps(manifest,indent=2)); print('DONE')


## 7 — Luki-only external transfer of frozen Musanga U-Net (no retraining)

**Run Section 3B first, then this cell only** after connecting to Colab with a GPU (T4/L4); it mounts Drive, imports dependencies, reads the `KVO_04` production manifest / exactly one frozen final ResNet18 U-Net checkpoint, and performs **Luki-only** probability inference. Does not execute any training, dataset creation or four-site map generation from earlier KVO_04 cells.

Source: `data/Geo_Ref/Luki.tif`; ROI mask: calculated directly from `Tree_Annotation_Luki.gpkg` via Section 3B (10 m grouping / 5 m buffer); does not read an existing ROI file. Output in the **same directory as other sites**: `KVO_work/KVO_04_final_multisite/probability_maps/Luki_musanga_probability_uint16.tif`. Encoding is identical to Baego: UInt16 [0, 10000] represents probability [0,1], and 65535 means NoData. The GeoTIFF retains Luki's full image dimensions/georeferencing and uses sparse tiling so only ROI-intersecting windows require model inference. The 4-band Luki image uses its first three bands as RGB; the existing alpha/data mask is retained.

**Important:** the Luki annotation layer is **not used to train or calibrate the model**. The generated ROI polygons are derived from the same annotations, so this is a spatially restricted out-of-site transfer map, not an annotation-independent choice of evaluation locations. The existing KVO_04 model was trained on four sites *excluding Luki*. Check source RGB order and radiometry before ecological interpretations.

In [ ]:
# KVO_04 / BLOCK 7 — Frozen Musanga U-Net probabilities for Luki ONLY.
# Self-contained GPU inference cell. Never trains and never updates other site rasters.
import sys,subprocess,json,hashlib,shutil,time,gc
from pathlib import Path
import numpy as np,geopandas as gpd,pandas as pd,rasterio
from rasterio.windows import Window,from_bounds
from rasterio.features import rasterize
from shapely.geometry import box
from IPython.display import display
from tqdm.auto import tqdm
from google.colab import drive
import torch

# ---- Paths: authoritative KVO_04 model and same multi-site probability folder ----
MY71=Path("/content/drive/MyDrive")
if not MY71.exists():drive.mount("/content/drive")
ROOT72=MY71/"Colab Notebooks"/"Musanga"
GEO72=ROOT72/"data"/"Geo_Ref"
LUKI72=GEO72/"Luki.tif"
ROI72=GEO72/"Annotations"/"Tree_Annotation_Luki.gpkg"   # source crowns, NOT an ROI
OUT72=ROOT72/"KVO_work"/"KVO_04_final_multisite"
MAPS72=OUT72/"probability_maps"
MAPS72.mkdir(parents=True,exist_ok=True)
FINAL72=MAPS72/"Luki_musanga_probability_uint16.tif"
QA72=OUT72/"Luki_probability_transfer_QA.json"
for p72 in (LUKI72,ROI72):
    assert p72.exists(),f"Missing source input {p72}."
assert "aggregate_annotation_rois" in globals(), "Run Section 3B before Luki-only inference."
assert torch.cuda.is_available(),"Use a Colab T4 or L4 GPU for Luki U-Net inference."

try:
    import segmentation_models_pytorch as smp
except ImportError:
    subprocess.run([sys.executable,"-m","pip","-q","install","segmentation-models-pytorch"],check=True)
    import segmentation_models_pytorch as smp
DEVICE72="cuda"
# Exact authoritative original normalization and inference context.
MEAN72=np.array([.485,.456,.406],np.float32)[:,None,None]
STD72=np.array([.229,.224,.225],np.float32)[:,None,None]
CORE72=512;HALO72=64;SCALE72=10000;NODATA72=65535

manifest72=OUT72/"production_manifest.json"
if manifest72.exists():
    manifest_info72=json.loads(manifest72.read_text())
    selected72=Path(manifest_info72["model"])
    if not selected72.exists():selected72=OUT72/selected72.name
    assert selected72.exists(),f"Checkpoint referenced by production_manifest.json missing: {selected72}"
else:
    model_candidates72=sorted(OUT72.glob("final_unet_resnet18_*.pt"))
    assert len(model_candidates72)==1,(
        "Cannot identify ONE authoritative checkpoint from KVO_04: "
        f"found {len(model_candidates72)} files. Provide production_manifest.json or "
        "select the verified final checkpoint explicitly. Never choose a checkpoint arbitrarily.")
    selected72=model_candidates72[0]
print("FROZEN checkpoint:",selected72)
print("Checkpoint size (MB):",round(selected72.stat().st_size/1e6,2))
assert selected72.stat().st_size>1_000_000,"Suspiciously small checkpoint."

# Build precisely matching U-Net with no internet pretrained-weight downloads.
model72=smp.Unet(encoder_name="resnet18",encoder_weights=None,in_channels=3,classes=1).to(DEVICE72)
weights72=torch.load(selected72,map_location="cpu",weights_only=True)
assert isinstance(weights72,dict)
model72.load_state_dict(weights72,strict=True)
del weights72
model72.eval()
print("Model loaded, strict state_dict=True; training disabled.")

roi72=aggregate_annotation_rois("Luki",export=True)
assert len(roi72)>0 and roi72.crs is not None,"Computed Luki ROI invalid."
with rasterio.open(LUKI72) as src72:
    assert src72.count>=3 and src72.crs is not None
    assert src72.dtypes[0]=="uint8",(
        f"KVO_04 training assumes RGB UInt8 0–255; Luki input is {src72.dtypes}. "
        "Do not rescale silently; inspect radiometry before transfer.")
    roi72=roi72.to_crs(src72.crs)
    footprint72=box(*src72.bounds)
    roi72=roi72[roi72.geometry.intersects(footprint72)].copy()
    assert len(roi72)>0,"Luki ROI does not overlap TIFF."
    geometries72=list(roi72.geometry)
    width72,height72=src72.width,src72.height
    crs72,transform72=src72.crs,src72.transform
    print("Luki image:",width72,"x",height72,"bands:",src72.count,
          "pixel size:",src72.res,"ROIs:",len(geometries72))
    # Only windows that intersect ROI bounds. Avoid iterating 1.4 billion pixels.
    tile_set72=set()
    for g in geometries72:
        clip72=g.intersection(footprint72)
        win72=from_bounds(*clip72.bounds,transform=transform72)
        start_r72=max(0,int(np.floor(win72.row_off/CORE72)))
        end_r72=min(int(np.ceil(height72/CORE72))-1,
                    int(np.floor((win72.row_off+win72.height-1e-7)/CORE72)))
        start_c72=max(0,int(np.floor(win72.col_off/CORE72)))
        end_c72=min(int(np.ceil(width72/CORE72))-1,
                    int(np.floor((win72.col_off+win72.width-1e-7)/CORE72)))
        for rr72 in range(start_r72,end_r72+1):
            for cc72 in range(start_c72,end_c72+1):
                tile_set72.add((rr72*CORE72,cc72*CORE72))
    tiles72=sorted(tile_set72)
    assert tiles72,f"No Luki tiles intersect ROI."
    print("Candidate ROI tiles:",len(tiles72),
          "| all mosaic tiles:",int(np.ceil(height72/CORE72)*np.ceil(width72/CORE72)))

# Protect preexisting user output. If already present, verify first and stop.
if FINAL72.exists():
    with rasterio.open(FINAL72) as existing72:
        ok72=(existing72.width==width72 and existing72.height==height72 and
              existing72.crs==crs72 and existing72.transform==transform72 and
              existing72.count==1 and existing72.dtypes[0]=="uint16" and
              existing72.nodata==NODATA72 and
              existing72.tags().get("MODEL_CHECKPOINT")==selected72.name)
    if ok72:
        print("Existing validated Luki probability map; SKIPPING inference:",FINAL72)
    else:
        raise RuntimeError(f"Existing output has unexpected schema or model: {FINAL72}. "
                           "Do not overwrite automatically; inspect and rename manually.")
else:
    # Stage locally for speed; sparse unallocated tiles must equal declared NoData.
    temp72=Path("/content/Luki_musanga_probability_uint16_partial.tif")
    temp72.unlink(missing_ok=True)
    validpix72=0;roi_valid72=0;inferred_tiles72=0;probs72=[]
    with rasterio.open(LUKI72) as src72:
        profile72=src72.profile.copy()
        profile72.update(driver="GTiff",count=1,dtype="uint16",nodata=NODATA72,
                        compress="deflate",predictor=2,tiled=True,
                        blockxsize=256,blockysize=256,bigtiff="YES",
                        SPARSE_OK="TRUE",interleave="band")
        with rasterio.open(temp72,"w",**profile72) as dst72:
            # Verify that unwritten sparse tiles resolve to NoData, not probability zero.
            corner72=dst72.read(1,window=Window(0,0,min(32,width72),min(32,height72)))
            assert np.all(corner72==NODATA72),(
                "Sparse GeoTIFF doesn't initialize unwritten pixels to NoData here. "
                "Stop rather than generating a raster with false zero probabilities.")
            for r0,c0 in tqdm(tiles72,desc="Luki frozen U-Net ROI tiles"):
                hh=min(CORE72,height72-r0);ww=min(CORE72,width72-c0)
                cw=Window(c0,r0,ww,hh)
                cb=box(*rasterio.windows.bounds(cw,src72.transform))
                neighbors=[g for g in geometries72 if g.intersects(cb)]
                if not neighbors:continue
                inside=rasterize([(g,1) for g in neighbors],out_shape=(hh,ww),
                    transform=src72.window_transform(cw),dtype="uint8",fill=0)>0
                if not inside.any():continue
                source_valid=src72.dataset_mask(window=cw)>0
                valid=inside&source_valid
                if not valid.any():continue
                win=Window(c0-HALO72,r0-HALO72,ww+2*HALO72,hh+2*HALO72)
                a=src72.read([1,2,3],window=win,boundless=True,fill_value=0)
                assert a.dtype==np.uint8
                a=a.astype(np.float32)/255.
                ph=(-a.shape[1])%32;pw=(-a.shape[2])%32
                if ph or pw:a=np.pad(a,((0,0),(0,ph),(0,pw)))
                x=torch.from_numpy((a-MEAN72)/STD72).unsqueeze(0).to(DEVICE72)
                with torch.inference_mode(),torch.amp.autocast("cuda"):
                    pred=torch.sigmoid(model72(x).squeeze()).float().cpu().numpy()
                p=pred[:int(win.height),:int(win.width)][HALO72:HALO72+hh,HALO72:HALO72+ww]
                out=np.full((hh,ww),NODATA72,dtype=np.uint16)
                out[valid]=np.rint(np.clip(p[valid],0.,1.)*SCALE72).astype(np.uint16)
                dst72.write(out,1,window=cw)
                roi_valid72+=int(valid.sum())
                inferred_tiles72+=1
                if len(probs72)<25000:probs72.extend(out[valid].ravel()[::max(1,int(valid.sum()/256))].tolist())
            dst72.update_tags(MODEL_CHECKPOINT=selected72.name,
                              MUSANGA_SCALE=SCALE72,TRAINING_SITES="MOS_1_2,Yoko,Baego,Yangambi",
                              ROI_MASK="10m connected crown annotation components + 5m buffer",INFERENCE="KVO_04 exact 512 core/64 halo, RGB first 3, ImageNet normalize",
                              LUKI_ONLY="true",PROVENANCE="frozen model, zero Luki training labels")
    # Validation before permanent copy.
    with rasterio.open(temp72) as pr72:
        assert pr72.crs==crs72 and pr72.transform==transform72
        assert pr72.width==width72 and pr72.height==height72
        assert pr72.dtypes[0]=="uint16" and pr72.nodata==NODATA72
        assert pr72.tags()["MODEL_CHECKPOINT"]==selected72.name
        # Whole-image corners outside candidate windows must remain NoData.
        for row,col in ((0,0),(height72-1,width72-1)):
            assert int(pr72.read(1,window=Window(col,row,1,1))[0,0])==NODATA72 or (row,col) in [
                (r,c) for r,c in tiles72], "Unexpected values in unwritten sparse area"
    assert inferred_tiles72>0 and roi_valid72>0,"No pixels inferred; inspect ROI and source mask."
    # Copy to existing four-site folder, preserving previous site maps.
    shutil.copy2(temp72,FINAL72)
    temp72.unlink(missing_ok=True)
    quality72={"site":"Luki","source_tif":str(LUKI72),"roi_source_annotation":str(ROI72),
               "model":str(selected72),"output":str(FINAL72),
               "inferred_tiles":inferred_tiles72,"valid_pixels":roi_valid72,
               "image_width":width72,"image_height":height72,"prob_scale":SCALE72,
               "nodata":NODATA72,
               "sample_mean_probability":float(np.mean(probs72)/SCALE72) if probs72 else None}
    QA72.write_text(json.dumps(quality72,indent=2))
    print("LUKI INFERENCE COMPLETE:",FINAL72)
    display(pd.DataFrame([quality72]))
# Standalone integrity check even when map was previously completed.
with rasterio.open(FINAL72) as o72:
    print("Luki map:",o72.width,"x",o72.height,"CRS:",o72.crs,
          "dtype:",o72.dtypes[0],"NoData:",o72.nodata,"size GB:",round(FINAL72.stat().st_size/1e9,3))
print("Other sites' probability rasters were not touched.")
